In this exercise, you will perform prompt engineering on a dialogue summarization task using [Flan-T5](https://huggingface.co/google/flan-t5-large) and the [dialogsum dataset](https://huggingface.co/datasets/knkarthick/dialogsum). You will explore how different prompts affect the output of the model, and compare zero-shot and few-shot inferences. <br/>
Complete the code in the cells below.

### 1. Set up Required Dependencies

In [1]:
!pip install datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cud

In [2]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig
from datasets import load_dataset

### 2. Explore the Dataset

In [3]:
from datasets import load_dataset
dataset = load_dataset('knkarthick/dialogsum')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Print several dialogues with their baseline summaries.

In [4]:
example_indices = [0, 42, 800]
dash_line = '-' * 100

for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example', i + 1)
    print(dash_line)
    print('INPUT DIALOGUE:')
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print('BASELINE HUMAN SUMMARY:')
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

----------------------------------------------------------------------------------------------------
Example 1
----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to 

This dataset contains examples of workplace dialogues, each followed by a human-written summary that captures the essence of the conversation. The structure includes a multi-turn exchange between participants (e.g., "#Person1#" and "#Person2#") and a corresponding summary that highlights the key points discussed.

### 3. Summarize Dialogues without Prompt Engineering

Load the Flan-T5-large model and its tokenizer.

In [5]:
model_name = 'google/flan-t5-large'

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

**Exercise**: Use the pre-trained model to summarize the example dialogues without any prompt engineering. Use the `model.generate()` function with `max_new_tokens=50`.

In [6]:
### WRITE YOUR CODE HERE
for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example', i + 1)
    print(dash_line)
    print('INPUT DIALOGUE:')
    print(dataset['test'][index]['dialogue'])
    print(dash_line)

    # Generate Model Summary
    inputs = tokenizer(
        dataset['test'][index]['dialogue'],
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    summary_ids = model.generate(inputs["input_ids"], max_new_tokens=50)
    model_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    print('BASELINE HUMAN SUMMARY:')
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print('MODEL GENERATED SUMMARY:')
    print(model_summary)
    print(dash_line)
    print()

----------------------------------------------------------------------------------------------------
Example 1
----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to 

You can see that the model generations make some sense, but the model doesn't seem to be sure what task it is supposed to accomplish and it often just makes up the next sentence in the dialogue. Prompt engineering can help here.

### 4. Summarize Dialogues with Instruction Prompts

In order to instruct the model to perform a task (e.g., summarize a dialogue), you can take the dialogue and convert it into an instruction prompt. This is often called **zero-shot inference**.

**Exercise**: Wrap the dialogues in a descriptive instruction (e.g., "Summarize the following conversation."), and examine how the generated text changes.

In [7]:
### WRITE YOUR CODE HERE
for i, index in enumerate(example_indices):
    print(dash_line)
    print(f'Example {i+1}')
    print(dash_line)

    dialogue = dataset['test'][index]['dialogue']
    human_summary = dataset['test'][index]['summary']

    # Create instruction prompt
    instruction_prompt = f"Summarize the following conversation.\n\n{dialogue}"

    # Generate with instruction
    inputs = tokenizer(instruction_prompt, return_tensors="pt", truncation=True, max_length=1024)
    summary_ids = model.generate(inputs["input_ids"], max_new_tokens=50)
    instructed_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # Display results
    print('INPUT DIALOGUE:')
    print(dialogue)
    print(dash_line)
    print('BASELINE HUMAN SUMMARY:')
    print(human_summary)
    print(dash_line)
    print('MODEL SUMMARY (WITH INSTRUCTION):')
    print(instructed_summary)
    print(dash_line)
    print()


----------------------------------------------------------------------------------------------------
Example 1
----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to 

This is much better! But the model still does not pick up on the nuance of the conversations though.

**Exercise:** Experiment with the prompt text and see how it influences the generated output. Do the inferences change if you end the prompt with just empty string vs. `Summary: `?

In [8]:
### WRITE YOUR CODE HERE
prompt_variations = {
    'empty_suffix': "Summarize the following conversation.\n\n{dialogue}",
    'summary_prefix': "Summarize the following conversation.\n\n{dialogue}\nSummary:"
}

for i, index in enumerate(example_indices):
    print(dash_line)
    print(f'Example {i+1}')
    print(dash_line)
    dialogue = dataset['test'][index]['dialogue']

    # Generate with both prompt types
    results = {}
    for prompt_name, prompt_template in prompt_variations.items():
        full_prompt = prompt_template.format(dialogue=dialogue)
        inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=1024)
        summary_ids = model.generate(inputs["input_ids"], max_new_tokens=50)
        results[prompt_name] = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # Display comparisons
    print('INPUT DIALOGUE:')
    print(dialogue)
    print(dash_line)
    print('PROMPT WITH EMPTY ENDING:')
    print(results['empty_suffix'])
    print(dash_line)
    print('PROMPT ENDING WITH "Summary:":')
    print(results['summary_prefix'])
    print(dash_line)
    print("\n\n")


----------------------------------------------------------------------------------------------------
Example 1
----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to 

While the "Summary:" suffix nudges the model toward abstraction, both outputs fail to match the human summary’s conciseness and holistic understanding. The 50-token limit prematurely truncates critical information about warnings/termination.

**Exercise:** Flan-T5 has many prompt templates that are published for certain tasks [here](https://github.com/google-research/FLAN/blob/main/flan/v2/templates.py). Try using its pre-built prompts for dialogue summarization (e.g., the ones under the `"samsum"` key) and see how they influence the outputs.


In [9]:
# Samsum prompt templates (filtered for summarization)
summarization_prompts = [
    ("{dialogue}\n\nBriefly summarize that dialogue.", "{summary}"),
    ("Here is a dialogue:\n{dialogue}\n\nWrite a short summary!", "{summary}"),
    ("Dialogue:\n{dialogue}\n\nWhat is a summary of this dialogue?", "{summary}"),
    ("{dialogue}\n\nWhat was that dialogue about, in two sentences or less?", "{summary}"),
    ("Here is a dialogue:\n{dialogue}\n\nWhat were they talking about?", "{summary}"),
    ("Dialogue:\n{dialogue}\nWhat were the main points in that conversation?", "{summary}"),
    ("Dialogue:\n{dialogue}\nWhat was going on in that conversation?", "{summary}"),
]

example_indices = [0, 42, 800]
dash_line = '-' * 100

for idx in example_indices:
    dialogue = dataset['test'][idx]['dialogue']
    human_summary = dataset['test'][idx]['summary']

    print(dash_line)
    print(f"DIALOGUE:\n{dialogue}")
    print(dash_line)
    print(f"HUMAN SUMMARY:\n{human_summary}")
    print(dash_line)

    # Test all prompt variations
    for i, (prompt_template, _) in enumerate(summarization_prompts):
        # Format prompt
        formatted_prompt = prompt_template.format(dialogue=dialogue)

        # Generate summary
        inputs = tokenizer(formatted_prompt, return_tensors="pt", truncation=True, max_length=512)
        outputs = model.generate(
            inputs.input_ids,
            max_new_tokens=50,
            num_beams=4,
            early_stopping=True
        )
        generated_summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

        print(f"\nPROMPT {i+1} ({prompt_template.split('{')[0].strip()}):")
        print(generated_summary)

    print("\n" + "="*100 + "\n")

----------------------------------------------------------------------------------------------------
DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want an

Notice that the prompts from Flan-T5 did help, but the model still struggles to pick up on the nuance of the conversation in some cases. This is what you will try to solve with few-shot inferencing.

### 5. Summarize Dialogues with a Few-Shot Inference

**Few-shot inference** is the practice of providing an LLM with several examples of prompt-response pairs that match your task - before your actual prompt that you want completed. This is called "in-context learning" and puts your model into a state that understands your specific task.

**Exercise:** Build a function that takes a list of `in_context_example_indexes`, generates a prompt with the examples, then at the end appends the prompt that you want the model to complete (`test_example_index`). Use the same Flan-T5 prompt template from Section 3. Make sure to separate between the examples with `"\n\n\n"`.

In [10]:
def make_prompt(in_context_example_indices, test_example_index):
    ### WRITE YOUR CODE HERE
    prompt_parts = []

    # Add in-context examples from dataset
    for idx in in_context_example_indices:
        example = dataset['test'][idx]
        prompt_parts.append(
            f"Input: {example['dialogue']}\n"
            f"Summary: {example['summary']}"
        )

    # Add test example (without summary)
    test_dialogue = dataset['test'][test_example_index]['dialogue']
    prompt_parts.append(
        f"Input: {test_dialogue}\n"
        "Summary:"
    )

    # Join with triple newline separators
    prompt = "\n\n\n".join(prompt_parts)






    return prompt

In [13]:
in_context_example_indices = [0, 10, 20]
test_example_index = 800

few_shot_prompt = make_prompt(in_context_example_indices, test_example_index)
print(few_shot_prompt)

Input: #Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue with the m

Now pass this prompt to the model perform a few shot inference:

In [11]:
### WRITE YOUR CODE HERE
def perform_few_shot_inference(in_context_indices, test_index):
    # Generate the few-shot prompt
    prompt = make_prompt(in_context_indices, test_index)

    # Tokenize and generate summary
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        inputs.input_ids,
        max_new_tokens=50,
        num_beams=4,
        early_stopping=True
    )

    # Decode and clean output
    generated_summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Get ground truth for comparison
    human_summary = dataset['test'][test_index]['summary']
    input_dialogue = dataset['test'][test_index]['dialogue']

    # Print results
    print("="*100)
    print("INPUT DIALOGUE:")
    print(input_dialogue)
    print("-"*100)
    print("HUMAN SUMMARY:")
    print(human_summary)
    print("-"*100)
    print("MODEL SUMMARY:")
    print(generated_summary)
    print("="*100)

# Example usage
perform_few_shot_inference(
    in_context_indices=[42, 800],  # Few-shot examples
    test_index=0                    # Dialogue to summarize
)

INPUT DIALOGUE:
#Person1#: Ms. Dawson, I need you to take a dictation for me.
#Person2#: Yes, sir...
#Person1#: This should go out as an intra-office memorandum to all employees by this afternoon. Are you ready?
#Person2#: Yes, sir. Go ahead.
#Person1#: Attention all staff... Effective immediately, all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited.
#Person2#: Sir, does this apply to intra-office communications only? Or will it also restrict external communications?
#Person1#: It should apply to all communications, not only in this office between employees, but also any outside communications.
#Person2#: But sir, many employees use Instant Messaging to communicate with their clients.
#Person1#: They will just have to change their communication methods. I don't want any - one using Instant Messaging in this office. It wastes too much time! Now, please continue w

**Exercise:** Experiment with the few-shot inferencing:
- Choose different dialogues - change the indices in the `in_context_example_indices` list and `test_example_index` value.
- Change the number of examples. Be sure to stay within the model's 512 context length, however.

How well does few-shot inference work with other examples?

In [12]:
### WRITE YOUR CODE HERE
alt_example_indices = [1, 5, 7]
alt_test_index = 42

prompt = make_prompt(alt_example_indices, alt_test_index)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)
outputs = model.generate(**inputs, max_new_tokens=60)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Alternate Few-shot Summary:\n", summary)

Alternate Few-shot Summary:
 Summary: #Person2 got stuck in traffic again.


The provided examples demonstrate that few-shot inference works poorly when examples are irrelevant or mismatched, but can succeed with carefully curated, task-aligned examples.

### 6. Generative Configuration Parameters for Inference

You can change the configuration parameters of the `generate()` method to see a different output from the LLM. So far the only parameter that you have been setting was `max_new_tokens=50`, which defines the maximum number of tokens to generate. A convenient way of organizing the configuration parameters is to use `GenerationConfig` class. By setting the parameter `do_sample = True`, you can activate various decoding strategies which influence the next token from the probability distribution over the entire vocabulary. You can then adjust the outputs changing `temperature` and other parameters (such as `top_k` and `top_p`). A full list of available parameters can be found in the [Hugging Face Generation documentation](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationConfig).

**Exercise:** Change the configuration parameters to investigate their influence on the output. Analyze your results.

In [15]:
prompt_variations = [
    lambda d: f"{d}\n\nBriefly summarize that dialogue.",
    lambda d: f"Here is a dialogue:\n{d}\n\nWrite a short summary!",
    lambda d: f"Dialogue:\n{d}\n\nWhat is a summary of this dialogue?",
    lambda d: f"{d}\n\nWhat was that dialogue about, in two sentences or less?",
    lambda d: f"Here is a dialogue:\n{d}\n\nWhat were they talking about?",
    lambda d: f"Dialogue:\n{d}\nWhat were the main points in that conversation?",
    lambda d: f"Dialogue:\n{d}\nWhat was going on in that conversation?",
]
samsum_prompt = lambda d: f"Dialogue:\n{d}\n\nWhat is the summary of the dialogue above?"

In [16]:
### WRITE YOUR CODE HERE
from transformers import GenerationConfig

prompt = samsum_prompt(dataset['test'][0]['dialogue'])
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

# Try sampling with temperature and top_p
configs = [
    GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.7, top_p=0.9),
    GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0, top_k=50),
]

for i, config in enumerate(configs):
    outputs = model.generate(**inputs, generation_config=config)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nConfig {i+1} Output:\n{summary}\n")


Config 1 Output:
Person1 is dictating a memo to all employees to ban the use of instant messaging during working hours.


Config 2 Output:
Person1 uses an intra-office memo to communicate all office communications restricted to email correspondence and official memos to all employees in order to avoid the use of Instant Messaging during working hours. Only emails are acceptable documents and employees are prohibited from using Instant



The two generated summaries illustrate how different decoding strategies affect the language model's output. In **Config 1**, which uses a lower temperature (0.7) and top-p sampling (0.9), the summary is **concise and focused**. This reflects the model's tendency to favor more probable, conservative outputs with lower temperature settings. In contrast, **Config 2**, which uses a higher temperature (1.0) and top-k sampling (k=50), produces a **more detailed and verbose summary**. This shows that increasing the temperature and using top-k sampling encourages more diverse outputs but can lead to verbosity or less coherence, especially without careful control of max token limits.